# Preprocessing: updated census data (mifkad_2.csv)

Builds the join key `City_agas_code` and saves the canonical processed census file used by the corona/census merge step.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_PATH = Path("/home/bcrlab/igguest/porat_naama/data/raw/mifkad_2.csv")
OUT_DIR = Path("/home/bcrlab/igguest/porat_naama/data/processed/mifkad")

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Load updated census data
df_mifkad = pd.read_csv(RAW_PATH)
print(f"rows: {len(df_mifkad):,}, cols: {df_mifkad.shape[1]}")
df_mifkad.head()

rows: 3,857, cols: 70


,OBJECTID,SHEM_YISHUV_HEB,SHEM_YISHUV_ENG,SEMEL_YISHUV,YISHUV_STAT_2022,STAT_2022,Stat2022_Unite,Stat2022_Ref,Main_Function_Code,Main_Function_Txt,...,hh_MidatDatiyut,hh_MidatDatiyut_Name,Computer_avg,Vehicle0_pcnt,Vehicle2up_pcnt,Parking_pcnt,own_pcnt,rent_pcnt,Shape__Area,Shape__Length
0,1,שחר,SHAHAR,7,70001,1.0,1,NaN,1.0,מגורים,...,2.0,מסורתי,1.1,16.9,45.9,88.9,63.3,18.3,2.299948e+06,7470.551048
1,2,תירוש,TIROSH,10,100001,1.0,1,NaN,1.0,מגורים,...,3.0,דתי/ דתי מאוד,0.8,26.2,24.2,73.2,45.4,30.5,1.348877e+06,5373.979325
2,3,"ניר ח""ן",NIR HEN,11,110001,1.0,1,NaN,1.0,מגורים,...,1.0,חילוני,1.5,15.7,48.2,76.6,53.6,30.6,8.950495e+05,4668.721712
3,4,חצבה,HAZEVA,13,130001,1.0,1,NaN,1.0,מגורים,...,1.0,חילוני,1.9,11.1,60.7,95.5,63.5,26.9,1.306252e+06,5623.056457
4,5,נועם,NO'AM,15,150001,1.0,1,NaN,1.0,מגורים,...,2.0,מסורתי,0.7,8.2,39.3,82.0,29.5,40.3,1.656053e+06,6987.293605


In [4]:
# Clean key columns before building the join key
df_mifkad["SEMEL_YISHUV"] = df_mifkad["SEMEL_YISHUV"].astype(str).str.strip()
df_mifkad["STAT_2022"] = pd.to_numeric(df_mifkad["STAT_2022"], errors="coerce").astype("Int64")

In [5]:
# Build City_agas_code (matches the corona-file convention):
# localities with a single row (not split into sub-areas) get suffix "0",
# same for rows with no STAT_2022.
loc_counts = df_mifkad["SEMEL_YISHUV"].value_counts()
is_single_row = df_mifkad["SEMEL_YISHUV"].map(loc_counts).eq(1)

suffix = df_mifkad["STAT_2022"].astype(str)
suffix = suffix.mask(is_single_row | df_mifkad["STAT_2022"].isna(), "0")

df_mifkad["City_agas_code"] = df_mifkad["SEMEL_YISHUV"] + "_" + suffix

In [6]:
# Sanity checks on the new key before Porat's modifications
print(f"rows:                  {len(df_mifkad):,}")
print(f"single-row localities: {is_single_row.sum():,}")
print(f"missing STAT_2022:     {df_mifkad['STAT_2022'].isna().sum():,}")
print(f"keys ending in '_0':   {df_mifkad['City_agas_code'].str.endswith('_0').sum():,}")
print(f"key is unique:         {df_mifkad['City_agas_code'].is_unique}")

assert df_mifkad["City_agas_code"].notna().all(), "City_agas_code has missing values"
assert df_mifkad["City_agas_code"].is_unique, "City_agas_code is not unique"
assert (df_mifkad["City_agas_code"] == "_").sum() == 0, "Empty key parts found"

rows:                  3,857
single-row localities: 1,242
missing STAT_2022:     104
keys ending in '_0':   1,242
key is unique:         True


In [7]:
### Porat

# This cell creates a city-level X_0 row for every locality that does not already have one,
# while keeping track of which city rows were synthesized.

if df_mifkad.index.name == 'City_agas_code':
    df_mifkad = df_mifkad.reset_index()

df_mifkad['SEMEL_YISHUV'] = pd.to_numeric(
    df_mifkad['SEMEL_YISHUV'],
    errors='coerce'
).astype('Int64')

df_mifkad['STAT_2022'] = pd.to_numeric(
    df_mifkad['STAT_2022'],
    errors='coerce'
).astype('Int64')

df_mifkad['is_city_aggregate'] = 0
df_mifkad['is_unite_duplicate'] = 0

existing_city_codes = set(df_mifkad['City_agas_code'].dropna())

towns_missing_city_row = sorted(
    int(t)
    for t in df_mifkad['SEMEL_YISHUV'].dropna().unique()
    if f"{int(t)}_0" not in existing_city_codes
)

new_city_rows = []

for town_code in towns_missing_city_row:
    # np.nan (not pd.NA): keeps numeric columns as float64 after the concat below,
    # instead of silently downcasting them to object dtype (which breaks .corr() etc.)
    new_row = {col: np.nan for col in df_mifkad.columns}

    new_row['SEMEL_YISHUV'] = town_code
    new_row['STAT_2022'] = 0
    new_row['City_agas_code'] = f"{town_code}_0"
    new_row['is_city_aggregate'] = 1
    new_row['is_unite_duplicate'] = 0

    new_city_rows.append(new_row)

df_mifkad = pd.concat(
    [df_mifkad, pd.DataFrame(new_city_rows)],
    ignore_index=True
)

print(f"Towns missing X_0: {len(towns_missing_city_row)}")
print(f"X_0 rows created: {len(new_city_rows)}")

Towns missing X_0: 145
X_0 rows created: 145


In [8]:
### Porat

# This cell fills pop_approx and hh_total_approx for synthesized X_0 rows
# by summing the original non-zero statistical areas within each locality.

count_cols = ['pop_approx', 'hh_total_approx']

for col in count_cols:
    df_mifkad[col] = pd.to_numeric(df_mifkad[col], errors='coerce')

original_agas_rows = df_mifkad[
    (df_mifkad['STAT_2022'] != 0) &
    (df_mifkad['is_unite_duplicate'] == 0)
]

city_counts = (
    original_agas_rows
    .groupby('SEMEL_YISHUV')[count_cols]
    .sum(min_count=1)
)

city_mask = df_mifkad['is_city_aggregate'] == 1

for col in count_cols:
    df_mifkad.loc[city_mask, col] = (
        df_mifkad.loc[city_mask, 'SEMEL_YISHUV']
        .map(city_counts[col])
    )

print(
    df_mifkad.loc[
        city_mask,
        ['City_agas_code', 'pop_approx', 'hh_total_approx']
    ].head()
)

     City_agas_code  pop_approx  hh_total_approx
3857           28_0     15850.0           4490.0
3858           31_0     35350.0          10010.0
3859           53_0     10650.0           3310.0
3860           70_0    227850.0          73810.0
3861          154_0     14110.0           4180.0


In [9]:
### Porat

# This cell propagates combined Stat2022_Unite records to every statistical area they
# cover. Existing values are preserved: data are copied from the source row only when
# the target value is 0 or NaN and the source contains a meaningful non-zero value.

import re
import numbers

# Store the original row from which each duplicated statistical area was created
df_mifkad['unite_source_City_agas_code'] = pd.NA

# Only original, non-city rows should be considered as a duplication source
rows_to_expand = df_mifkad[
    (df_mifkad['STAT_2022'] != 0) &
    (df_mifkad['is_unite_duplicate'] == 0)
]

data_cols = [
    col for col in df_mifkad.columns
    if col not in (
        'City_agas_code',
        'SEMEL_YISHUV',
        'STAT_2022',
        'is_city_aggregate',
        'is_unite_duplicate',
        'unite_source_City_agas_code',
    )
]


def extract_unite_codes(value):
    if pd.isna(value):
        return []

    numbers_found = re.findall(r'\d+(?:\.\d+)?', str(value))
    return [int(float(x)) for x in numbers_found]


def is_zero_or_na(value):
    """True if a value is missing or numeric zero."""
    if pd.isna(value):
        return True

    return isinstance(value, numbers.Number) and value == 0


def is_meaningful(value):
    """True if a value contains non-missing, non-zero information."""
    return not is_zero_or_na(value)


existing_city_codes = set(df_mifkad['City_agas_code'])

filled_rows = 0
filled_values = 0
preserved_values = 0
appended_rows = []


for _, row in rows_to_expand.iterrows():

    unite_codes = extract_unite_codes(row['Stat2022_Unite'])

    # Only combined units containing more than one statistical area need expansion
    if len(unite_codes) <= 1:
        continue

    source_code = row['City_agas_code']
    town_code = int(row['SEMEL_YISHUV'])

    for stat_code in unite_codes:

        # Skip the source statistical area itself
        if stat_code == row['STAT_2022']:
            continue

        new_city_agas_code = f"{town_code}_{stat_code}"
        target_mask = df_mifkad['City_agas_code'] == new_city_agas_code

        if target_mask.any():

            row_was_filled = False

            # Fill each column separately.
            # Never overwrite an existing meaningful value.
            for col in data_cols:

                source_value = row[col]

                for idx in df_mifkad.index[target_mask]:

                    target_value = df_mifkad.at[idx, col]

                    if is_zero_or_na(target_value) and is_meaningful(source_value):
                        df_mifkad.at[idx, col] = source_value
                        filled_values += 1
                        row_was_filled = True

                    elif is_meaningful(target_value):
                        preserved_values += 1

            # Mark the row as belonging to the combined census unit
            df_mifkad.loc[target_mask, 'is_unite_duplicate'] = 1
            df_mifkad.loc[
                target_mask,
                'unite_source_City_agas_code'
            ] = source_code

            if row_was_filled:
                filled_rows += 1

        elif new_city_agas_code not in existing_city_codes:

            # No row exists at all, so create a complete copy
            new_row = row.copy()

            new_row['STAT_2022'] = stat_code
            new_row['City_agas_code'] = new_city_agas_code
            new_row['is_unite_duplicate'] = 1
            new_row['unite_source_City_agas_code'] = source_code

            appended_rows.append(new_row)
            existing_city_codes.add(new_city_agas_code)


if appended_rows:
    df_mifkad = pd.concat(
        [df_mifkad, pd.DataFrame(appended_rows)],
        ignore_index=True
    )


print(f"Stat2022_Unite existing rows filled: {filled_rows}")
print(f"Individual missing/zero values filled: {filled_values}")
print(f"Existing non-zero values preserved: {preserved_values}")
print(f"Rows appended (no existing row): {len(appended_rows)}")
print(f"New df_mifkad shape: {df_mifkad.shape}")

Stat2022_Unite existing rows filled: 240
Individual missing/zero values filled: 12235
Existing non-zero values preserved: 2509
Rows appended (no existing row): 0
New df_mifkad shape: (4002, 74)


In [10]:
### Porat

# This cell sorts the completed Mifkad dataset, restores City_agas_code as the index,
# and reports how many city-level and Stat2022_Unite duplicate rows were created.

df_mifkad = (
    df_mifkad
    .sort_values(['SEMEL_YISHUV', 'STAT_2022'])
    .set_index('City_agas_code')
)

print(f"Total rows: {len(df_mifkad):,}")
print(f"Synthesized X_0 rows: {df_mifkad['is_city_aggregate'].sum():,}")
print(f"Stat2022_Unite duplicates: {df_mifkad['is_unite_duplicate'].sum():,}")

df_mifkad.head()

Total rows: 4,002
Synthesized X_0 rows: 145
Stat2022_Unite duplicates: 240


,OBJECTID,SHEM_YISHUV_HEB,SHEM_YISHUV_ENG,SEMEL_YISHUV,YISHUV_STAT_2022,STAT_2022,Stat2022_Unite,Stat2022_Ref,Main_Function_Code,Main_Function_Txt,...,Vehicle0_pcnt,Vehicle2up_pcnt,Parking_pcnt,own_pcnt,rent_pcnt,Shape__Area,Shape__Length,is_city_aggregate,is_unite_duplicate,unite_source_City_agas_code
City_agas_code,,,,,,,,,,,,,,,,,,,,,
7_0,1.0,שחר,SHAHAR,7,70001.0,1,1,NaN,1.0,מגורים,...,16.9,45.9,88.9,63.3,18.3,2.299948e+06,7470.551048,0,0,<NA>
10_0,2.0,תירוש,TIROSH,10,100001.0,1,1,NaN,1.0,מגורים,...,26.2,24.2,73.2,45.4,30.5,1.348877e+06,5373.979325,0,0,<NA>
11_0,3.0,"ניר ח""ן",NIR HEN,11,110001.0,1,1,NaN,1.0,מגורים,...,15.7,48.2,76.6,53.6,30.6,8.950495e+05,4668.721712,0,0,<NA>
13_0,4.0,חצבה,HAZEVA,13,130001.0,1,1,NaN,1.0,מגורים,...,11.1,60.7,95.5,63.5,26.9,1.306252e+06,5623.056457,0,0,<NA>
15_0,5.0,נועם,NO'AM,15,150001.0,1,1,NaN,1.0,מגורים,...,8.2,39.3,82.0,29.5,40.3,1.656053e+06,6987.293605,0,0,<NA>


In [11]:
### Porat

# This cell constructs census values for synthesized X_0 city rows.
#
# IMPORTANT:
# - Only original statistical-area rows are used.
# - Stat2022_Unite duplicate rows are excluded.
# - Exact/defensible aggregations are separated from approximate ones.
# - Ratios that can be reconstructed are reconstructed rather than averaged.
# - Medians and other non-aggregatable variables are intentionally left unchanged.
# - Only synthesized city rows (is_city_aggregate == 1) are modified.


# ============================================================
# Settings
# ============================================================

# Minimum fraction of the relevant population/households that must have
# a non-missing value before we calculate a city aggregate.
#
# Example:
# If only areas representing 40% of Tel Aviv's population have a value,
# we do NOT want to present their weighted mean as the Tel Aviv value.
MIN_WEIGHT_COVERAGE = 0.90


# ============================================================
# Make City_agas_code a column temporarily if it is the index
# ============================================================

restore_index = df_mifkad.index.name == 'City_agas_code'

if restore_index:
    df_mifkad = df_mifkad.reset_index()


# ============================================================
# Basic validation
# ============================================================

required_cols = [
    'SEMEL_YISHUV',
    'STAT_2022',
    'is_city_aggregate',
    'is_unite_duplicate',
    'pop_approx',
    'hh_total_approx',
]

missing_required = [
    c for c in required_cols
    if c not in df_mifkad.columns
]

if missing_required:
    raise KeyError(
        f"Missing required columns: {missing_required}"
    )


# ============================================================
# Aggregation groups
# ============================================================

# ------------------------------------------------------------
# 1. Counts
# ------------------------------------------------------------

sum_cols = [
    'pop_approx',
    'hh_total_approx',
]


# ------------------------------------------------------------
# 2. Population-weighted:
#    relatively defensible because the denominator is population
# ------------------------------------------------------------

population_weighted_exact = [
    'inst_pcnt',
    'Foreign_pcnt',
    'age0_19_pcnt',
    'age20_64_pcnt',
    'age65_pcnt',
]


# ------------------------------------------------------------
# 3. Household-weighted:
#    denominator is households
# ------------------------------------------------------------

household_weighted_cols = [
    'size_avg',
    'hh0_5_pcnt',
    'hh18_24_pcnt',
    'Computer_avg',
    'Vehicle0_pcnt',
    'Vehicle2up_pcnt',
    'Parking_pcnt',
    'own_pcnt',
    'rent_pcnt',
]


# ------------------------------------------------------------
# 4. Population-weighted APPROXIMATIONS
#
# These variables actually refer to subgroups such as:
# ages 18-34, employed people, people aged 15+, women, etc.
#
# The exact subgroup denominator is not available in the current
# statistical-area table, so population weighting is used as an
# ML-oriented approximation.
# ------------------------------------------------------------

population_weighted_approx = [

    # Marriage
    'married18_34_pcnt',
    'married45_54_pcnt',

    # Israel / abroad / origin
    'j_isr_pcnt',
    'j_abr_pcnt',
    'aliya2002_pcnt',
    'aliya2010_pcnt',
    'israel_pcnt',
    'asia_pcnt',
    'africa_pcnt',
    'europe_pcnt',
    'america_pcnt',

    # Fertility / disability
    'ChldBorn_avg',
    'koshi5_pcnt',

    # Education / employment
    'AcadmCert_pcnt',
    'WrkY_pcnt',
    'Empl_pcnt',
    'SelfEmpl_pcnt',
    'HrsWrkWk_avg',
    'Wrk_15_17_pcnt',
    'WrkOutLoc_pcnt',

    # Income-distribution percentages
    'EmployeesWage_decile9Up',
    'SelfEmployedWage_decile9Up',
]


# ------------------------------------------------------------
# Columns intentionally NOT aggregated here
# ------------------------------------------------------------

do_not_aggregate = [

    # Medians cannot be reconstructed from area medians
    'age_median',
    'm_age_median',
    'w_age_median',
    'MarriageAge_mdn',
    'm_MarriageAge_mdn',
    'w_MarriageAge_mdn',
    'employeesAnnual_medWage',
    'SelfEmployedAnnual_medWage',

    # GIS perimeter cannot simply be summed
    'Shape__Length',
]


# Keep only columns actually present in the dataframe
sum_cols = [
    c for c in sum_cols
    if c in df_mifkad.columns
]

population_weighted_exact = [
    c for c in population_weighted_exact
    if c in df_mifkad.columns
]

population_weighted_approx = [
    c for c in population_weighted_approx
    if c in df_mifkad.columns
]

household_weighted_cols = [
    c for c in household_weighted_cols
    if c in df_mifkad.columns
]


# ============================================================
# Convert relevant columns to numeric
# ============================================================

extra_numeric_cols = [
    'STAT_2022',
    'sexRatio',
    'change_pcnt',
    'DependencyRatio',
    'koshi65_pcnt',
]

numeric_cols = list(dict.fromkeys(
    sum_cols +
    population_weighted_exact +
    population_weighted_approx +
    household_weighted_cols +
    [
        c for c in extra_numeric_cols
        if c in df_mifkad.columns
    ]
))

for col in numeric_cols:
    df_mifkad[col] = pd.to_numeric(
        df_mifkad[col],
        errors='coerce'
    )


# ============================================================
# Select ONLY original statistical-area records
# ============================================================

original_area_mask = (
    df_mifkad['STAT_2022'].notna() &
    (df_mifkad['STAT_2022'] != 0) &
    (df_mifkad['is_unite_duplicate'].fillna(0) == 0)
)

areas = df_mifkad.loc[original_area_mask].copy()


# ============================================================
# Helper functions
# ============================================================

def weighted_mean_with_coverage(
    group,
    value_col,
    weight_col,
    min_coverage=MIN_WEIGHT_COVERAGE
):
    """
    Weighted mean with a coverage check.

    Coverage =
        weight represented by rows with a valid value
        ------------------------------------------------
        total available positive weight

    Returns
    -------
    value, coverage
    """

    weights = pd.to_numeric(
        group[weight_col],
        errors='coerce'
    )

    values = pd.to_numeric(
        group[value_col],
        errors='coerce'
    )

    has_weight = (
        weights.notna() &
        (weights > 0)
    )

    total_weight = weights.loc[has_weight].sum()

    if pd.isna(total_weight) or total_weight <= 0:
        return pd.NA, 0.0

    valid = (
        has_weight &
        values.notna()
    )

    valid_weight = weights.loc[valid].sum()

    coverage = valid_weight / total_weight

    if coverage < min_coverage:
        return pd.NA, coverage

    result = (
        values.loc[valid] *
        weights.loc[valid]
    ).sum() / valid_weight

    return result, coverage


def reconstruct_sex_ratio(
    group,
    min_coverage=MIN_WEIGHT_COVERAGE
):
    """
    Reconstruct city sex ratio rather than averaging area ratios.

    Assumes:
        sexRatio = 100 * males / females
    """

    if (
        'sexRatio' not in group.columns or
        'pop_approx' not in group.columns
    ):
        return pd.NA, 0.0

    pop = pd.to_numeric(
        group['pop_approx'],
        errors='coerce'
    )

    ratio = pd.to_numeric(
        group['sexRatio'],
        errors='coerce'
    )

    has_pop = (
        pop.notna() &
        (pop > 0)
    )

    total_pop = pop.loc[has_pop].sum()

    if total_pop <= 0:
        return pd.NA, 0.0

    valid = (
        has_pop &
        ratio.notna() &
        (ratio >= 0)
    )

    covered_pop = pop.loc[valid].sum()
    coverage = covered_pop / total_pop

    if coverage < min_coverage:
        return pd.NA, coverage

    r = ratio.loc[valid] / 100

    female = (
        pop.loc[valid] /
        (1 + r)
    )

    male = (
        pop.loc[valid] -
        female
    )

    if female.sum() <= 0:
        return pd.NA, coverage

    city_ratio = (
        100 *
        male.sum() /
        female.sum()
    )

    return city_ratio, coverage


def reconstruct_change_pct(
    group,
    min_coverage=MIN_WEIGHT_COVERAGE
):
    """
    Reconstruct city population change rather than averaging
    statistical-area percentage changes.

    Assumes:
        change_pcnt =
        100 * (current_population - previous_population)
              / previous_population
    """

    if (
        'change_pcnt' not in group.columns or
        'pop_approx' not in group.columns
    ):
        return pd.NA, 0.0

    pop_now = pd.to_numeric(
        group['pop_approx'],
        errors='coerce'
    )

    change = pd.to_numeric(
        group['change_pcnt'],
        errors='coerce'
    )

    has_pop = (
        pop_now.notna() &
        (pop_now > 0)
    )

    total_pop = pop_now.loc[has_pop].sum()

    if total_pop <= 0:
        return pd.NA, 0.0

    rate = change / 100

    valid = (
        has_pop &
        change.notna() &
        ((1 + rate) > 0)
    )

    covered_pop = pop_now.loc[valid].sum()
    coverage = covered_pop / total_pop

    if coverage < min_coverage:
        return pd.NA, coverage

    previous_pop = (
        pop_now.loc[valid] /
        (1 + rate.loc[valid])
    )

    if previous_pop.sum() <= 0:
        return pd.NA, coverage

    city_change = (
        (
            pop_now.loc[valid].sum() /
            previous_pop.sum()
        ) - 1
    ) * 100

    return city_change, coverage


def first_non_null(series):
    """Return the first meaningful non-null value."""
    values = series.dropna()

    if len(values) == 0:
        return pd.NA

    return values.iloc[0]


# ============================================================
# Calculate city-level values
# ============================================================

city_aggregates = {}

diagnostics = []


for town_code, group in areas.groupby('SEMEL_YISHUV'):

    city_values = {}


    # --------------------------------------------------------
    # City names
    # --------------------------------------------------------

    for col in [
        'SHEM_YISHUV_HEB',
        'SHEM_YISHUV_ENG',
    ]:
        if col in group.columns:
            city_values[col] = first_non_null(
                group[col]
            )


    # --------------------------------------------------------
    # Counts
    # --------------------------------------------------------

    for col in sum_cols:

        values = pd.to_numeric(
            group[col],
            errors='coerce'
        )

        if values.notna().any():
            city_values[col] = values.sum(
                min_count=1
            )


    # --------------------------------------------------------
    # Population-weighted exact / defensible columns
    # --------------------------------------------------------

    for col in population_weighted_exact:

        value, coverage = weighted_mean_with_coverage(
            group,
            col,
            'pop_approx'
        )

        city_values[col] = value

        diagnostics.append({
            'SEMEL_YISHUV': town_code,
            'column': col,
            'method': 'population_weighted',
            'coverage': coverage,
            'calculated': pd.notna(value),
        })


    # --------------------------------------------------------
    # Household-weighted columns
    # --------------------------------------------------------

    for col in household_weighted_cols:

        value, coverage = weighted_mean_with_coverage(
            group,
            col,
            'hh_total_approx'
        )

        city_values[col] = value

        diagnostics.append({
            'SEMEL_YISHUV': town_code,
            'column': col,
            'method': 'household_weighted',
            'coverage': coverage,
            'calculated': pd.notna(value),
        })


    # --------------------------------------------------------
    # Population-weighted APPROXIMATE columns
    # --------------------------------------------------------

    for col in population_weighted_approx:

        value, coverage = weighted_mean_with_coverage(
            group,
            col,
            'pop_approx'
        )

        city_values[col] = value

        diagnostics.append({
            'SEMEL_YISHUV': town_code,
            'column': col,
            'method': 'population_weighted_APPROX',
            'coverage': coverage,
            'calculated': pd.notna(value),
        })


    # --------------------------------------------------------
    # koshi65_pcnt
    #
    # Correct denominator is people aged 65+.
    # We can estimate that denominator from:
    #
    # pop_approx * age65_pcnt / 100
    # --------------------------------------------------------

    if all(
        c in group.columns
        for c in [
            'koshi65_pcnt',
            'age65_pcnt',
            'pop_approx'
        ]
    ):

        group_k65 = group.copy()

        group_k65['_age65_population'] = (
            group_k65['pop_approx'] *
            group_k65['age65_pcnt'] /
            100
        )

        value, coverage = weighted_mean_with_coverage(
            group_k65,
            'koshi65_pcnt',
            '_age65_population'
        )

        city_values['koshi65_pcnt'] = value

        diagnostics.append({
            'SEMEL_YISHUV': town_code,
            'column': 'koshi65_pcnt',
            'method': 'age65_population_weighted',
            'coverage': coverage,
            'calculated': pd.notna(value),
        })


    # --------------------------------------------------------
    # DependencyRatio
    #
    # Reconstruct from aggregated city age percentages.
    # --------------------------------------------------------

    required_age_cols = [
        'age0_19_pcnt',
        'age20_64_pcnt',
        'age65_pcnt',
    ]

    if all(
        pd.notna(city_values.get(c))
        for c in required_age_cols
    ):

        working_age = city_values['age20_64_pcnt']

        if working_age > 0:

            city_values['DependencyRatio'] = (
                1000 *
                (
                    city_values['age0_19_pcnt'] +
                    city_values['age65_pcnt']
                ) /
                working_age
            )


    # --------------------------------------------------------
    # sexRatio
    # --------------------------------------------------------

    if 'sexRatio' in group.columns:

        value, coverage = reconstruct_sex_ratio(
            group
        )

        city_values['sexRatio'] = value

        diagnostics.append({
            'SEMEL_YISHUV': town_code,
            'column': 'sexRatio',
            'method': 'reconstructed',
            'coverage': coverage,
            'calculated': pd.notna(value),
        })


    # --------------------------------------------------------
    # change_pcnt
    # --------------------------------------------------------

    if 'change_pcnt' in group.columns:

        value, coverage = reconstruct_change_pct(
            group
        )

        city_values['change_pcnt'] = value

        diagnostics.append({
            'SEMEL_YISHUV': town_code,
            'column': 'change_pcnt',
            'method': 'reconstructed',
            'coverage': coverage,
            'calculated': pd.notna(value),
        })


    city_aggregates[town_code] = city_values


# ============================================================
# Fill synthesized X_0 rows
# ============================================================

city_mask = (
    (df_mifkad['STAT_2022'] == 0) &
    (df_mifkad['is_city_aggregate'] == 1)
)

filled_values = 0


for idx in df_mifkad.index[city_mask]:

    town_code = df_mifkad.at[
        idx,
        'SEMEL_YISHUV'
    ]

    if town_code not in city_aggregates:
        continue

    for col, new_value in city_aggregates[town_code].items():

        if col not in df_mifkad.columns:
            continue

        if pd.notna(new_value):

            # X_0 is synthesized, so this aggregation is authoritative.
            # Recalculate rather than preserving an older synthesized value.
            df_mifkad.at[idx, col] = new_value

            filled_values += 1


# ============================================================
# Diagnostics
# ============================================================

aggregation_diagnostics = pd.DataFrame(
    diagnostics
)


print("City aggregation completed.")
print()

print(
    f"Population-weighted exact columns:\n"
    f"{population_weighted_exact}"
)

print()

print(
    f"Household-weighted columns:\n"
    f"{household_weighted_cols}"
)

print()

print(
    f"Population-weighted approximate columns:\n"
    f"{population_weighted_approx}"
)

print()

print(
    f"Minimum required weight coverage: "
    f"{MIN_WEIGHT_COVERAGE:.0%}"
)

print(
    f"X_0 values written: "
    f"{filled_values:,}"
)


# Show variables rejected because too much source data was missing
if not aggregation_diagnostics.empty:

    low_coverage = aggregation_diagnostics[
        ~aggregation_diagnostics['calculated']
    ].copy()

    print(
        f"Aggregates skipped because of insufficient data: "
        f"{len(low_coverage):,}"
    )


# ============================================================
# Restore City_agas_code as index
# ============================================================

if restore_index:
    df_mifkad = df_mifkad.set_index(
        'City_agas_code'
    )

City aggregation completed.

Population-weighted exact columns:
['inst_pcnt', 'Foreign_pcnt', 'age0_19_pcnt', 'age20_64_pcnt', 'age65_pcnt']

Household-weighted columns:
['size_avg', 'hh0_5_pcnt', 'hh18_24_pcnt', 'Computer_avg', 'Vehicle0_pcnt', 'Vehicle2up_pcnt', 'Parking_pcnt', 'own_pcnt', 'rent_pcnt']

Population-weighted approximate columns:
['married18_34_pcnt', 'married45_54_pcnt', 'j_isr_pcnt', 'j_abr_pcnt', 'aliya2002_pcnt', 'aliya2010_pcnt', 'israel_pcnt', 'asia_pcnt', 'africa_pcnt', 'europe_pcnt', 'america_pcnt', 'ChldBorn_avg', 'koshi5_pcnt', 'AcadmCert_pcnt', 'WrkY_pcnt', 'Empl_pcnt', 'SelfEmpl_pcnt', 'HrsWrkWk_avg', 'Wrk_15_17_pcnt', 'WrkOutLoc_pcnt', 'EmployeesWage_decile9Up', 'SelfEmployedWage_decile9Up']

Minimum required weight coverage: 90%
X_0 values written: 5,351
Aggregates skipped because of insufficient data: 7,107


In [12]:
# Sanity checks on the new key after Porat's modifications
# (City_agas_code is now the index, not a column - see the previous cell)
print(f"rows:                  {len(df_mifkad):,}")
print(f"single-row localities: {is_single_row.sum():,}")
print(f"missing STAT_2022:     {df_mifkad['STAT_2022'].isna().sum():,}")
print(f"keys ending in '_0':   {df_mifkad.index.str.endswith('_0').sum():,}")
print(f"key is unique:         {df_mifkad.index.is_unique}")

assert df_mifkad.index.notna().all(), "City_agas_code has missing values"
assert df_mifkad.index.is_unique, "City_agas_code is not unique"
assert (df_mifkad.index == "_").sum() == 0, "Empty key parts found"

rows:                  4,002
single-row localities: 1,242
missing STAT_2022:     104
keys ending in '_0':   1,387
key is unique:         True


In [13]:
# Quick look at the indexed dataset (City_agas_code was already set as the index
# by the earlier Porat sort/index cell)
print("Missing pop_approx:", df_mifkad["pop_approx"].isna().sum())
df_mifkad.info()

Missing pop_approx: 382
<class 'pandas.core.frame.DataFrame'>
Index: 4002 entries, 7_0 to 9975_0
Data columns (total 73 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   OBJECTID                     3857 non-null   float64
 1   SHEM_YISHUV_HEB              4002 non-null   object 
 2   SHEM_YISHUV_ENG              3944 non-null   object 
 3   SEMEL_YISHUV                 4002 non-null   Int64  
 4   YISHUV_STAT_2022             3857 non-null   float64
 5   STAT_2022                    3898 non-null   Int64  
 6   Stat2022_Unite               3475 non-null   object 
 7   Stat2022_Ref                 240 non-null    float64
 8   Main_Function_Code           3697 non-null   float64
 9   Main_Function_Txt            3697 non-null   object 
 10  ROVA                         1420 non-null   float64
 11  TAT_ROVA                     2080 non-null   float64
 12  Religion_Stat_Code           3475 non-null   float64


In [14]:
# Save an inspection copy (raw structure) and the canonical processed file
df_mifkad.to_csv(OUT_DIR / "raw_mif_2_inspect.csv", encoding="utf-8-sig")
df_mifkad.to_csv(OUT_DIR / "Mifkad_2_processed.csv")

print("Saved:", OUT_DIR / "raw_mif_2_inspect.csv")
print("Saved:", OUT_DIR / "Mifkad_2_processed.csv")

Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/raw_mif_2_inspect.csv
Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/Mifkad_2_processed.csv


In [15]:
df_mifkad.head()

,OBJECTID,SHEM_YISHUV_HEB,SHEM_YISHUV_ENG,SEMEL_YISHUV,YISHUV_STAT_2022,STAT_2022,Stat2022_Unite,Stat2022_Ref,Main_Function_Code,Main_Function_Txt,...,Vehicle0_pcnt,Vehicle2up_pcnt,Parking_pcnt,own_pcnt,rent_pcnt,Shape__Area,Shape__Length,is_city_aggregate,is_unite_duplicate,unite_source_City_agas_code
City_agas_code,,,,,,,,,,,,,,,,,,,,,
7_0,1.0,שחר,SHAHAR,7,70001.0,1,1,NaN,1.0,מגורים,...,16.9,45.9,88.9,63.3,18.3,2.299948e+06,7470.551048,0,0,<NA>
10_0,2.0,תירוש,TIROSH,10,100001.0,1,1,NaN,1.0,מגורים,...,26.2,24.2,73.2,45.4,30.5,1.348877e+06,5373.979325,0,0,<NA>
11_0,3.0,"ניר ח""ן",NIR HEN,11,110001.0,1,1,NaN,1.0,מגורים,...,15.7,48.2,76.6,53.6,30.6,8.950495e+05,4668.721712,0,0,<NA>
13_0,4.0,חצבה,HAZEVA,13,130001.0,1,1,NaN,1.0,מגורים,...,11.1,60.7,95.5,63.5,26.9,1.306252e+06,5623.056457,0,0,<NA>
15_0,5.0,נועם,NO'AM,15,150001.0,1,1,NaN,1.0,מגורים,...,8.2,39.3,82.0,29.5,40.3,1.656053e+06,6987.293605,0,0,<NA>


In [16]:
with pd.option_context('display.max_rows', None):
    display(df_mifkad.loc[['31_4', '31_5']].T)

City_agas_code,31_4,31_5
OBJECTID,24.0,25.0
SHEM_YISHUV_HEB,אופקים,אופקים
SHEM_YISHUV_ENG,OFAQIM,OFAQIM
SEMEL_YISHUV,31,31
YISHUV_STAT_2022,310004.0,310005.0
STAT_2022,4,5
Stat2022_Unite,4+5,4+5
Stat2022_Ref,NaN,4.0
Main_Function_Code,1.0,2.0
Main_Function_Txt,מגורים,תעשיה


In [17]:
with pd.option_context('display.max_rows', None):
    display(df_mifkad.loc[['5000_0', '5000_111']].T)

City_agas_code,5000_0,5000_111
OBJECTID,NaN,2605.0
SHEM_YISHUV_HEB,תל אביב -יפו,תל אביב -יפו
SHEM_YISHUV_ENG,TEL AVIV - YAFO,TEL AVIV - YAFO
SEMEL_YISHUV,5000,5000
YISHUV_STAT_2022,NaN,50000111.0
STAT_2022,0,111
Stat2022_Unite,NaN,111
Stat2022_Ref,NaN,NaN
Main_Function_Code,NaN,1.0
Main_Function_Txt,NaN,מגורים


## Feature reduction: representative + composite index per domain

For each correlated block of raw columns, keep one raw representative column
(for single-feature tests) plus one PCA-based composite index (for
combined-feature tests). Outputs (table, figures, manifest) are all saved
under `data/processed/mifkad/`.

In [18]:
import matplotlib.pyplot as plt

# Domain -> source columns, chosen raw representative, and the shared idea
DOMAINS = {
    "age_structure": {
        "cols": ["age0_19_pcnt", "age20_64_pcnt", "age65_pcnt", "DependencyRatio",
                 "age_median", "m_age_median", "w_age_median"],
        "representative": "age65_pcnt",
        "idea": "Overall age skew of the area (young vs. old population).",
        "idea_he": "שיפוע הגילאים באזור (צעיר מול מבוגר).",
    },
    "origin": {
        "cols": ["j_isr_pcnt", "j_abr_pcnt", "aliya2002_pcnt", "aliya2010_pcnt",
                 "israel_pcnt", "asia_pcnt", "africa_pcnt", "europe_pcnt", "america_pcnt"],
        "representative": "j_isr_pcnt",
        "idea": "Veteran-Israeli vs. immigrant / continent-of-origin composition.",
        "idea_he": "הרכב הוותק/מוצא של התושבים (ישראלי-ותיק מול עולה, ולפי יבשת מוצא).",
    },
    "household_family": {
        "cols": ["married18_34_pcnt", "married45_54_pcnt", "MarriageAge_mdn",
                 "m_MarriageAge_mdn", "w_MarriageAge_mdn", "ChldBorn_avg",
                 "size_avg", "hh0_5_pcnt", "hh18_24_pcnt"],
        "representative": "size_avg",
        "idea": "Family life-cycle stage (young families vs. older/smaller households).",
        "idea_he": "שלב מחזור החיים המשפחתי (משפחות צעירות מול משקי בית מבוגרים/קטנים).",
    },
    # "ses" was split into employment / income / housing_assets (CBS's own
    # grouping): the merged block had no representative above mean|r|=0.5.
    "employment": {
        "cols": ["WrkY_pcnt", "Empl_pcnt", "SelfEmpl_pcnt", "HrsWrkWk_avg", "Wrk_15_17_pcnt"],
        "representative": "WrkY_pcnt",
        "idea": "Employment level and work pattern of the area.",
        "idea_he": "רמת התעסוקה ודפוס העבודה באזור.",
    },
    "income": {
        "cols": ["employeesAnnual_medWage", "EmployeesWage_decile9Up",
                 "SelfEmployedAnnual_medWage", "SelfEmployedWage_decile9Up"],
        "representative": "employeesAnnual_medWage",
        "idea": "Wage level of the area (salaried and self-employed).",
        "idea_he": "רמת השכר באזור (שכירים ועצמאים).",
    },
    "housing_assets": {
        "cols": ["Computer_avg", "Vehicle0_pcnt", "Vehicle2up_pcnt", "Parking_pcnt",
                 "own_pcnt", "rent_pcnt"],
        "representative": "Vehicle2up_pcnt",
        "idea": "Household asset ownership and tenure (vehicles, parking, home ownership).",
        "idea_he": "בעלות על נכסים וחזקה בדיור (רכב, חניה, בעלות על דירה).",
    },
    "disability": {
        "cols": ["koshi5_pcnt", "koshi65_pcnt"],
        "representative": "koshi5_pcnt",
        "idea": "Functional-difficulty (disability) burden in the area.",
        "idea_he": "נטל קושי תפקודי (נכות) באזור.",
    },
}

# AcadmCert_pcnt (education) is a single-column CBS group - no block to reduce,
# kept as a standalone raw feature, not part of DOMAINS.

In [19]:
def pca_first_component(df, cols):
    """PC1 (z-scored, rows with any missing value in cols dropped) of df[cols].
    Returns (scores aligned to df.index, explained variance ratio of PC1)."""
    sub = df[cols].dropna()
    z = (sub - sub.mean()) / sub.std(ddof=0)
    _, s, vt = np.linalg.svd(z.values, full_matrices=False)
    pc1 = pd.Series(z.values @ vt[0], index=sub.index)
    explained_ratio = (s[0] ** 2) / (s ** 2).sum()
    return pc1.reindex(df.index), explained_ratio


# Running list of (path, description, description_he) for every file this notebook saves
output_log = []

def log_output(path, description, description_he):
    """Register a saved output file for the manifest and print its path."""
    output_log.append({
        "file_path": str(path),
        "contents": description,
        "contents_he": description_he,
    })
    print("Saved:", path)


# Register the two files already saved earlier in this notebook
log_output(
    OUT_DIR / "Mifkad_2_processed.csv",
    "Processed census data (mifkad_2.csv), indexed by City_agas_code.",
    "קובץ נתוני המפקד המעודכן (mifkad_2.csv) אחרי בניית מפתח הצירוף City_agas_code "
    "(סמל יישוב + מספר אזור סטטיסטי, עם סיומת '0' ליישובים שאינם מחולקים לאזורים). "
    "מכיל את כל 70 העמודות המקוריות, מאונדקס לפי המפתח, ומשמש כקובץ הבסיס למיזוג "
    "עם נתוני הקורונה.",
)
log_output(
    OUT_DIR / "raw_mif_2_inspect.csv",
    "Same content as Mifkad_2_processed.csv, utf-8-sig encoding for manual inspection.",
    "עותק זהה לתוכן של Mifkad_2_processed.csv, נשמר בקידוד utf-8-sig כדי שייפתח "
    "נכון באקסל עם טקסט עברי. מיועד לבדיקה ידנית בלבד, לא לשימוש בהמשך הצנרת.",
)

Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/Mifkad_2_processed.csv
Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/raw_mif_2_inspect.csv


In [20]:
description_rows = []
domain_corr = {}  # domain -> Series of |correlation| between representative and each other column

for domain, spec in DOMAINS.items():
    cols, rep, idea = spec["cols"], spec["representative"], spec["idea"]

    # how well the representative correlates with the rest of its block
    corr_with_rep = df_mifkad[cols].corr()[rep].drop(rep)
    domain_corr[domain] = corr_with_rep

    # composite index: PC1 of the whole block
    idx_name = f"{domain}_idx"
    pc1, explained_ratio = pca_first_component(df_mifkad, cols)
    df_mifkad[idx_name] = pc1
    idx_vs_rep_corr = df_mifkad[[idx_name, rep]].corr().iloc[0, 1]

    description_rows.append({
        "column_name": rep, "domain": domain, "type": "raw_representative",
        "source_columns": ";".join(cols), "idea": idea,
        "explained_variance_pct": np.nan,
        "mean_abs_corr_with_group": round(corr_with_rep.abs().mean(), 3),
    })
    description_rows.append({
        "column_name": idx_name, "domain": domain, "type": "composite_index",
        "source_columns": ";".join(cols),
        "idea": f"PC1 of the {domain} block; combined signal beyond the single representative.",
        "explained_variance_pct": round(explained_ratio * 100, 1),
        "mean_abs_corr_with_group": round(abs(idx_vs_rep_corr), 3),
    })

feature_description = pd.DataFrame(description_rows)
feature_description

,column_name,domain,type,source_columns,idea,explained_variance_pct,mean_abs_corr_with_group
0,age65_pcnt,age_structure,raw_representative,age0_19_pcnt;age20_64_pcnt;age65_pcnt;Dependen...,Overall age skew of the area (young vs. old po...,NaN,0.554
1,age_structure_idx,age_structure,composite_index,age0_19_pcnt;age20_64_pcnt;age65_pcnt;Dependen...,PC1 of the age_structure block; combined signa...,69.8,0.809
2,j_isr_pcnt,origin,raw_representative,j_isr_pcnt;j_abr_pcnt;aliya2002_pcnt;aliya2010...,Veteran-Israeli vs. immigrant / continent-of-o...,NaN,0.458
3,origin_idx,origin,composite_index,j_isr_pcnt;j_abr_pcnt;aliya2002_pcnt;aliya2010...,PC1 of the origin block; combined signal beyon...,43.7,0.991
4,size_avg,household_family,raw_representative,married18_34_pcnt;married45_54_pcnt;MarriageAg...,Family life-cycle stage (young families vs. ol...,NaN,0.625
5,household_family_idx,household_family,composite_index,married18_34_pcnt;married45_54_pcnt;MarriageAg...,PC1 of the household_family block; combined si...,60.6,0.856
6,WrkY_pcnt,employment,raw_representative,WrkY_pcnt;Empl_pcnt;SelfEmpl_pcnt;HrsWrkWk_avg...,Employment level and work pattern of the area.,NaN,0.399
7,employment_idx,employment,composite_index,WrkY_pcnt;Empl_pcnt;SelfEmpl_pcnt;HrsWrkWk_avg...,PC1 of the employment block; combined signal b...,45.7,0.772
8,employeesAnnual_medWage,income,raw_representative,employeesAnnual_medWage;EmployeesWage_decile9U...,Wage level of the area (salaried and self-empl...,NaN,0.564
9,income_idx,income,composite_index,employeesAnnual_medWage;EmployeesWage_decile9U...,PC1 of the income block; combined signal beyon...,64.3,0.864


In [21]:
# Save the table describing every new column (representative + composite index)
desc_path = OUT_DIR / "reduced_features_description.csv"
feature_description.to_csv(desc_path, index=False, encoding="utf-8-sig")
log_output(
    desc_path,
    "Representative + composite-index columns per domain: source columns and rationale.",
    "טבלה שמתעדת את כל עמודות צמצום המימדים שנוספו: לכל תחום (גיל, מוצא, משפחה, "
    "תעסוקה, הכנסה, דיור ורכוש, נכות) יש שורה לעמודת הנציג הגולמי ושורה לעמודת "
    "האינדקס המשולב (PCA) - כולל אילו עמודות מקור נכנסו לכל אחת, כמה אחוז מהשונות "
    "האינדקס מסביר, וכמה חזק הנציג מתואם עם שאר עמודות הקבוצה שלו.",
)

Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/reduced_features_description.csv


### Figures: does the representative column represent its group?

One bar chart per domain: correlation of the representative column with every
other raw column in its block. Dashed lines mark |r| = 0.7.

In [22]:
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
BAR_COLOR = "#2a78d6"

for domain, corr in domain_corr.items():
    rep = DOMAINS[domain]["representative"]
    idea_he = DOMAINS[domain]["idea_he"]
    corr = corr.sort_values()

    fig, ax = plt.subplots(figsize=(6.5, 0.4 * len(corr) + 1.5))
    ax.barh(corr.index, corr.values, color=BAR_COLOR)
    ax.axvline(0.7, color="gray", linestyle="--", linewidth=1)
    ax.axvline(-0.7, color="gray", linestyle="--", linewidth=1)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-1, 1)
    ax.set_xlabel(f"correlation with representative ({rep})")
    ax.set_title(f"{domain}: representative vs. rest of the group")
    fig.tight_layout()

    fig_path = FIG_DIR / f"{domain}_representative_corr.png"
    fig.savefig(fig_path, dpi=150)
    plt.close(fig)
    log_output(
        fig_path,
        f"Bar chart: correlation of the {domain} representative "
        f"column ({rep}) with the rest of its group.",
        f"גרף עמודות אופקי: {idea_he} מקדם המתאם של עמודת הנציג ({rep}) מול כל אחת "
        f"מיתר עמודות תחום ה-{domain}, בטווח 1- עד 1. קווים מקווקווים ב-0.7± מסמנים "
        f"סף מתאם חזק - מראה עד כמה הנציג באמת מייצג את שאר העמודות בקבוצה.",
    )

Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/figures/age_structure_representative_corr.png
Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/figures/origin_representative_corr.png
Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/figures/household_family_representative_corr.png
Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/figures/employment_representative_corr.png


Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/figures/income_representative_corr.png
Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/figures/housing_assets_representative_corr.png
Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/figures/disability_representative_corr.png


### Save the reduced dataset

Actual per-area values for the representative + composite-index columns
(the description table above documents them, but doesn't contain the data).

In [23]:
# Standalone raw columns: not part of any correlated domain block
STANDALONE_COLS = [
    "pop_approx", "hh_total_approx", "pop_density", "sexRatio", "inst_pcnt",
    "Foreign_pcnt", "change_pcnt", "WrkOutLoc_pcnt", "AcadmCert_pcnt",
    "Religion_Stat_Code", "Religion_Stat_Txt", "hh_MidatDatiyut", "hh_MidatDatiyut_Name",
]
representative_cols = [spec["representative"] for spec in DOMAINS.values()]
index_cols = [f"{domain}_idx" for domain in DOMAINS]

reduced_cols = STANDALONE_COLS + representative_cols + index_cols
df_reduced = df_mifkad[reduced_cols].copy()

print(f"reduced dataset: {df_reduced.shape[1]} columns (vs. 70 raw census columns)")
df_reduced.head(20)

reduced dataset: 27 columns (vs. 70 raw census columns)


,pop_approx,hh_total_approx,pop_density,sexRatio,inst_pcnt,Foreign_pcnt,change_pcnt,WrkOutLoc_pcnt,AcadmCert_pcnt,Religion_Stat_Code,...,employeesAnnual_medWage,Vehicle2up_pcnt,koshi5_pcnt,age_structure_idx,origin_idx,household_family_idx,employment_idx,income_idx,housing_assets_idx,disability_idx
City_agas_code,,,,,,,,,,,,,,,,,,,,,
7_0,840.0,220.0,1817.4,114.300000,NaN,11.1,5.0,69.200000,21.300000,1.0,...,130600.0,45.900000,9.90000,-0.074719,-1.152801,0.080718,-0.370945,0.185317,1.107866,-0.800392
10_0,490.0,160.0,1248.7,145.900000,NaN,8.6,6.0,72.900000,10.900000,1.0,...,90800.0,24.200000,7.60000,-1.239717,-1.116325,1.209631,-0.856161,-0.928215,-0.815924,0.137465
11_0,680.0,230.0,1692.9,103.500000,NaN,7.8,49.2,85.300000,38.200000,1.0,...,136100.0,48.200000,7.40000,-0.384428,-0.582944,-1.234492,-0.276899,-0.310009,0.813242,0.429530
13_0,1190.0,210.0,1862.1,197.600000,3.5,40.2,61.0,30.800000,24.700000,1.0,...,129000.0,60.700000,1.90000,-0.617045,-1.450474,-1.621878,-4.340977,-1.223441,2.182220,2.313529
15_0,480.0,140.0,1255.2,101.100000,NaN,2.3,34.3,83.900000,9.600000,1.0,...,101800.0,39.300000,9.10000,0.382539,-1.039986,1.259506,-0.227155,0.989683,-0.333988,0.535099
16_0,650.0,210.0,2400.3,85.700000,NaN,0.5,134.1,66.400000,40.100000,1.0,...,168900.0,52.900000,6.30000,0.207503,-1.708565,-1.848794,-0.947845,-4.151855,1.849966,0.534776
18_0,670.0,210.0,1810.8,103.500000,NaN,7.2,6.9,68.200000,34.600000,1.0,...,117700.0,54.400000,4.20000,-0.087105,-1.566702,-1.042198,-3.527519,-0.401012,1.550842,1.570776
21_0,1300.0,340.0,5897.9,100.300000,3.8,0.5,1229.6,78.800000,26.400000,1.0,...,158200.0,66.800000,4.10000,2.908095,-1.434502,-1.436405,-1.992602,-2.277946,2.181148,0.526274
22_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
reduced_path = OUT_DIR / "Mifkad_2_reduced.csv"
df_reduced.to_csv(reduced_path, encoding="utf-8-sig")
log_output(
    reduced_path,
    f"Reduced feature set ({df_reduced.shape[1]} columns, from 70 raw census columns): "
    "standalone raw variables + one representative and one composite index per domain. "
    "This is the file meant for the ML model, unlike reduced_features_description.csv "
    "which only documents the columns.",
    f"קובץ הנתונים המצומצם בפועל ({df_reduced.shape[1]} עמודות במקום 70 המקוריות): "
    "עמודות גולמיות עצמאיות (שלא היו חלק מאף בלוק מתואם) + עמודת נציג אחת ועמודת "
    "אינדקס משולב (PCA) אחת לכל תחום (גיל, מוצא, משפחה, תעסוקה, הכנסה, דיור ורכוש, "
    "נכות). זהו קובץ הנתונים המיועד לשימוש במודל ה-ML, בניגוד לטבלת התיעוד "
    "(reduced_features_description.csv) שרק מסבירה את העמודות בלי להכיל את הערכים.",
)

Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/Mifkad_2_reduced.csv


### Output files manifest

One row per file this notebook has saved: path + what it contains.

In [25]:
manifest_path = OUT_DIR / "output_files_manifest.csv"
manifest = pd.DataFrame(output_log)
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

print("Saved:", manifest_path)
manifest

Saved: /home/bcrlab/igguest/porat_naama/data/processed/mifkad/output_files_manifest.csv


,file_path,contents,contents_he
0,/home/bcrlab/igguest/porat_naama/data/processe...,"Processed census data (mifkad_2.csv), indexed ...",קובץ נתוני המפקד המעודכן (mifkad_2.csv) אחרי ב...
1,/home/bcrlab/igguest/porat_naama/data/processe...,"Same content as Mifkad_2_processed.csv, utf-8-...","עותק זהה לתוכן של Mifkad_2_processed.csv, נשמר..."
2,/home/bcrlab/igguest/porat_naama/data/processe...,Representative + composite-index columns per d...,טבלה שמתעדת את כל עמודות צמצום המימדים שנוספו:...
3,/home/bcrlab/igguest/porat_naama/data/processe...,Bar chart: correlation of the age_structure re...,גרף עמודות אופקי: שיפוע הגילאים באזור (צעיר מו...
4,/home/bcrlab/igguest/porat_naama/data/processe...,Bar chart: correlation of the origin represent...,גרף עמודות אופקי: הרכב הוותק/מוצא של התושבים (...
5,/home/bcrlab/igguest/porat_naama/data/processe...,Bar chart: correlation of the household_family...,גרף עמודות אופקי: שלב מחזור החיים המשפחתי (משפ...
6,/home/bcrlab/igguest/porat_naama/data/processe...,Bar chart: correlation of the employment repre...,גרף עמודות אופקי: רמת התעסוקה ודפוס העבודה באז...
7,/home/bcrlab/igguest/porat_naama/data/processe...,Bar chart: correlation of the income represent...,גרף עמודות אופקי: רמת השכר באזור (שכירים ועצמא...
8,/home/bcrlab/igguest/porat_naama/data/processe...,Bar chart: correlation of the housing_assets r...,גרף עמודות אופקי: בעלות על נכסים וחזקה בדיור (...
9,/home/bcrlab/igguest/porat_naama/data/processe...,Bar chart: correlation of the disability repre...,גרף עמודות אופקי: נטל קושי תפקודי (נכות) באזור...


In [27]:
df_reduced[df_reduced.index == '5000_114'].head()

,pop_approx,hh_total_approx,pop_density,sexRatio,inst_pcnt,Foreign_pcnt,change_pcnt,WrkOutLoc_pcnt,AcadmCert_pcnt,Religion_Stat_Code,...,employeesAnnual_medWage,Vehicle2up_pcnt,koshi5_pcnt,age_structure_idx,origin_idx,household_family_idx,employment_idx,income_idx,housing_assets_idx,disability_idx
City_agas_code,,,,,,,,,,,,,,,,,,,,,
5000_114,6230.0,1710.0,NaN,79.3,NaN,NaN,NaN,32.3,48.6,1.0,...,123300.0,26.9,4.7,-0.058675,-0.080474,-2.914544,-0.219755,-0.799222,-1.462342,0.836633
